In [ ]:
# ============================================================
# MODULE 2: CUSTOMER SEGMENTATION USING K-MEANS CLUSTERING
# THIRANEX DATA SCIENCE INTERNSHIP
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# ------------------------------------------------------------
# 1. CREATE SAMPLE SALES DATASET
# ------------------------------------------------------------

np.random.seed(42)

n = 120

categories = ["Technology", "Furniture", "Office Supplies"]
regions = ["North", "South", "East", "West"]
segments = ["Consumer", "Corporate", "Home Office"]
products = ["Laptop", "Chair", "Printer", "Table", "Phone", "Notebook"]
ship_modes = ["Standard", "Second Class", "First Class"]

sales_df = pd.DataFrame({
    "Order_ID": range(1001, 1001 + n),
    "Order_Date": pd.date_range("2025-01-01", periods=n, freq="D"),
    "Customer_ID": np.random.choice(
        [f"CUST_{i:03d}" for i in range(1, 31)], n
    ),
    "Category": np.random.choice(categories, n),
    "Product": np.random.choice(products, n),
    "Region": np.random.choice(regions, n),
    "Segment": np.random.choice(segments, n),
    "Ship_Mode": np.random.choice(ship_modes, n),
    "Quantity": np.random.randint(1, 10, n),
    "Sales": np.random.uniform(100, 5000, n).round(2)
})

sales_df["Profit"] = (
    sales_df["Sales"] * np.random.uniform(0.05, 0.30, n)
).round(2)

print("=" * 60)
print("CUSTOMER SEGMENTATION USING K-MEANS")
print("=" * 60)

print("\nDataset Shape:", sales_df.shape)

display(sales_df.head())

# ------------------------------------------------------------
# 2. CREATE CUSTOMER SUMMARY
# ------------------------------------------------------------

customer_summary = sales_df.groupby(
    ["Customer_ID", "Segment"]
).agg(
    Total_Sales=("Sales", "sum"),
    Total_Profit=("Profit", "sum"),
    Total_Quantity=("Quantity", "sum")
).reset_index()

print("\n" + "=" * 60)
print("CUSTOMER SUMMARY")
print("=" * 60)

display(customer_summary.head(10))

print("\nNumber of Customers:", len(customer_summary))

# ------------------------------------------------------------
# 3. SELECT FEATURES
# ------------------------------------------------------------

features = customer_summary[
    ["Total_Sales", "Total_Profit", "Total_Quantity"]
]

print("\nFeatures selected for clustering:")
print([
    "Total_Sales",
    "Total_Profit",
    "Total_Quantity"
])

# ------------------------------------------------------------
# 4. STANDARDIZE FEATURES
# ------------------------------------------------------------

scaler = StandardScaler()

scaled_features = scaler.fit_transform(features)

print("\nFeature standardization completed successfully.")

# ------------------------------------------------------------
# 5. APPLY K-MEANS CLUSTERING
# ------------------------------------------------------------

kmeans = KMeans(
    n_clusters=3,
    random_state=42,
    n_init=10
)

customer_summary["Cluster"] = kmeans.fit_predict(
    scaled_features
)

print("\nK-Means clustering completed successfully.")

# ------------------------------------------------------------
# 6. DISPLAY CLUSTER RESULTS
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("CUSTOMER SEGMENTATION RESULTS")
print("=" * 60)

display(customer_summary)

# ------------------------------------------------------------
# 7. CLUSTER SUMMARY
# ------------------------------------------------------------

cluster_summary = customer_summary.groupby(
    "Cluster"
).agg(
    Number_of_Customers=("Customer_ID", "count"),
    Average_Sales=("Total_Sales", "mean"),
    Average_Profit=("Total_Profit", "mean"),
    Average_Quantity=("Total_Quantity", "mean")
).reset_index()

print("\n" + "=" * 60)
print("CLUSTER SUMMARY")
print("=" * 60)

display(cluster_summary)

# ------------------------------------------------------------
# 8. SAVE CUSTOMER SEGMENTATION RESULTS
# ------------------------------------------------------------

customer_summary.to_csv(
    "customer_segmentation_results.csv",
    index=False
)

cluster_summary.to_csv(
    "cluster_summary.csv",
    index=False
)

print("\nCSV files created:")
print("✓ customer_segmentation_results.csv")
print("✓ cluster_summary.csv")

# ------------------------------------------------------------
# 9. SALES VS PROFIT CLUSTER VISUALIZATION
# ------------------------------------------------------------

plt.figure(figsize=(9, 6))

plt.scatter(
    customer_summary["Total_Sales"],
    customer_summary["Total_Profit"],
    c=customer_summary["Cluster"],
    s=80
)

plt.title("Customer Segmentation using K-Means")
plt.xlabel("Total Sales")
plt.ylabel("Total Profit")

plt.grid(True)
plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 10. SALES VS QUANTITY VISUALIZATION
# ------------------------------------------------------------

plt.figure(figsize=(9, 6))

plt.scatter(
    customer_summary["Total_Sales"],
    customer_summary["Total_Quantity"],
    c=customer_summary["Cluster"],
    s=80
)

plt.title("Customer Clusters: Sales vs Quantity")
plt.xlabel("Total Sales")
plt.ylabel("Total Quantity")

plt.grid(True)
plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 11. CLUSTER DISTRIBUTION
# ------------------------------------------------------------

cluster_counts = customer_summary["Cluster"].value_counts().sort_index()

plt.figure(figsize=(7, 5))

plt.bar(
    cluster_counts.index.astype(str),
    cluster_counts.values
)

plt.title("Number of Customers in Each Cluster")
plt.xlabel("Cluster")
plt.ylabel("Number of Customers")

plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 12. FINAL INSIGHTS
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("KEY INSIGHTS")
print("=" * 60)

for _, row in cluster_summary.iterrows():

    print(
        f"\nCluster {int(row['Cluster'])}: "
        f"{int(row['Number_of_Customers'])} customers | "
        f"Average Sales = {row['Average_Sales']:.2f} | "
        f"Average Profit = {row['Average_Profit']:.2f} | "
        f"Average Quantity = {row['Average_Quantity']:.2f}"
    )

print("\n" + "=" * 60)
print("MODULE 2 COMPLETED SUCCESSFULLY")
print("=" * 60)

print("\nGenerated Files:")
print("✓ customer_segmentation_results.csv")
print("✓ cluster_summary.csv")